In [ ]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [ ]:
# BB84 Quantum Key Distribution — Plain (No Attacker)

This notebook simulates the BB84 protocol between **Alice** (sender) and **Bob** (receiver), with no eavesdropper.

### Protocol overview
1. Alice picks random bits and random bases (rectilinear `+` or diagonal `×`), encodes each bit as a qubit.
2. Bob picks random bases and measures each qubit.
3. Alice and Bob publicly compare bases; they keep only the bits where they chose the same basis → **sifted key**.
4. They sacrifice a small sample of the sifted key to check for errors. With no attacker the error rate should be 0.

All randomness is generated by measuring qubits in the state |+⟩ = (|0⟩+|1⟩)/√2.

In [ ]:
%pip install qiskit==1.2.4 qiskit-aer==0.15.1 pylatexenc==2.10 -q

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
import math

# ── Shared simulator ────────────────────────────────────────────────────────
simulator = AerSimulator()

# ─────────────────────────────────────────────────────────────────────────────
# UTILITY: quantum random-bit generator
# Measures n qubits each prepared in |+⟩ = H|0⟩ to get n unbiased random bits.
# This is the ONLY source of randomness used in the protocol.
# ─────────────────────────────────────────────────────────────────────────────
def quantum_random_bits(n: int) -> list[int]:
    """Return a list of n random bits by measuring n qubits in the |+⟩ state."""
    qc = QuantumCircuit(n, n)
    qc.h(range(n))          # H|0⟩ = |+⟩  →  50/50 superposition
    qc.measure(range(n), range(n))
    job = simulator.run(transpile(qc, simulator), shots=1, memory=True)
    result_str = job.result().get_memory()[0]  # e.g. '01101'
    # Qiskit returns bits in little-endian order: reverse for natural indexing
    return [int(b) for b in reversed(result_str)]

print("Utility ready. Sample of 8 quantum random bits:", quantum_random_bits(8))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ALICE — encoding
# Basis convention: 0 → rectilinear (+), 1 → diagonal (×)
# Bit/basis encoding:
#   basis 0, bit 0 → |0⟩
#   basis 0, bit 1 → |1⟩
#   basis 1, bit 0 → |+⟩  (H|0⟩)
#   basis 1, bit 1 → |−⟩  (H|1⟩)
# ─────────────────────────────────────────────────────────────────────────────
N_QUBITS = 100   # number of qubits sent in the raw transmission
SAMPLE_FRACTION = 0.2   # fraction of sifted key used for error checking

def alice_encode(bits: list[int], bases: list[int]) -> list[QuantumCircuit]:
    """Alice encodes each bit in the chosen basis and returns a list of single-qubit circuits."""
    circuits = []
    for bit, basis in zip(bits, bases):
        qc = QuantumCircuit(1, 1)
        if bit == 1:
            qc.x(0)           # |1⟩
        if basis == 1:
            qc.h(0)           # rotate to diagonal basis
        circuits.append(qc)
    return circuits

# ── Alice generates her secret bits and random bases ────────────────────────
alice_bits  = quantum_random_bits(N_QUBITS)
alice_bases = quantum_random_bits(N_QUBITS)

alice_circuits = alice_encode(alice_bits, alice_bases)

print(f"Alice prepared {N_QUBITS} qubits.")
print(f"Alice bits  (first 20): {alice_bits[:20]}")
print(f"Alice bases (first 20): {alice_bases[:20]}  (0=+, 1=×)")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# BOB — measurement
# Bob independently picks random bases and measures each qubit Alice sent.
# ─────────────────────────────────────────────────────────────────────────────
def bob_measure(circuits: list[QuantumCircuit], bases: list[int]) -> list[int]:
    """Bob appends his basis choice and measures each qubit. Returns his result bits."""
    results = []
    for qc, basis in zip(circuits, bases):
        meas_qc = qc.copy()
        if basis == 1:
            meas_qc.h(0)      # rotate back from diagonal basis before measuring
        meas_qc.measure(0, 0)
        job = simulator.run(transpile(meas_qc, simulator), shots=1, memory=True)
        bit = int(job.result().get_memory()[0])
        results.append(bit)
    return results

bob_bases = quantum_random_bits(N_QUBITS)
bob_bits  = bob_measure(alice_circuits, bob_bases)

print(f"Bob measured {N_QUBITS} qubits.")
print(f"Bob bases (first 20): {bob_bases[:20]}  (0=+, 1=×)")
print(f"Bob bits  (first 20): {bob_bits[:20]}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SIFTING — Alice and Bob publicly compare bases (NOT bits)
# They keep only the positions where their bases matched.
# ─────────────────────────────────────────────────────────────────────────────
def sift_key(alice_bases, bob_bases, alice_bits, bob_bits):
    """Return sifted key bits for both parties (indices where bases agree)."""
    alice_sifted, bob_sifted, positions = [], [], []
    for i, (ab, bb) in enumerate(zip(alice_bases, bob_bases)):
        if ab == bb:
            alice_sifted.append(alice_bits[i])
            bob_sifted.append(bob_bits[i])
            positions.append(i)
    return alice_sifted, bob_sifted, positions

alice_sifted, bob_sifted, matching_positions = sift_key(
    alice_bases, bob_bases, alice_bits, bob_bits
)

print(f"Sifted key length: {len(alice_sifted)}  (expected ≈ {N_QUBITS//2})")
print(f"Alice sifted (first 20): {alice_sifted[:20]}")
print(f"Bob   sifted (first 20): {bob_sifted[:20]}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ERROR CHECKING — Alice and Bob publicly reveal a sample of their sifted bits
# A mismatch indicates noise or an eavesdropper.
# They discard the sample and keep the rest as the final secret key.
# ─────────────────────────────────────────────────────────────────────────────
DETECTION_THRESHOLD = 0.10   # flag an attack if error rate exceeds 10%

sample_size = max(1, int(len(alice_sifted) * SAMPLE_FRACTION))
sample_alice = alice_sifted[:sample_size]
sample_bob   = bob_sifted[:sample_size]

errors = sum(a != b for a, b in zip(sample_alice, sample_bob))
error_rate = errors / sample_size

print("── Error checking ──────────────────────────────────")
print(f"  Sample size : {sample_size} bits")
print(f"  Errors found: {errors}")
print(f"  Error rate  : {error_rate:.1%}")
print(f"  Threshold   : {DETECTION_THRESHOLD:.0%}")

if error_rate > DETECTION_THRESHOLD:
    print("  ⚠️  ATTACK DETECTED — aborting key exchange!")
else:
    print("  ✅ No attack detected.")
    final_key_alice = alice_sifted[sample_size:]
    final_key_bob   = bob_sifted[sample_size:]
    assert final_key_alice == final_key_bob, "Keys do not match — something is wrong!"
    print(f"  Final key length: {len(final_key_alice)} bits")
    print(f"  Final key (first 20 bits): {final_key_alice[:20]}")